# Legal Retrieval Model Comparison & Evaluation
## Research Notebook -- Full Comparative Study

**Models**: MiniLM | InLegalBERT | LegalBERT | BGE
**Retrievers**: Dense (x4) | GraphRAG-Augmented (x3)
**Metrics**: P@k | R@k | MRR | MAP | NDCG@k | F1@k | Cosine Distributions | Statistical Significance

> Run all cells top-to-bottom. Outputs saved to `data/experiments/` and `figures/`.

---
## Section 1 - Setup & Data Loading

In [1]:
# ============================================================
# SECTION 1 -- SETUP & DATA LOADING
# ============================================================
import os, json, time, math, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
from scipy.stats import wilcoxon
import faiss
import torch
import nltk
import networkx as nx

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

PROJECT_ROOT  = Path(r"E:\Legal Finale")
DATA_DIR      = PROJECT_ROOT / "data"
EXP_DIR       = DATA_DIR / "experiments"
EMBS_DIR      = DATA_DIR / "embeddings"
FIGS_DIR      = PROJECT_ROOT / "figures"
CHUNKS_FILE   = DATA_DIR / "chunks" / "legal_chunks.json"
LABELS_FILE   = EXP_DIR / "completed_labels.csv"

for d in [EXP_DIR, EMBS_DIR, FIGS_DIR, EXP_DIR/"LegalBERT", EXP_DIR/"GraphRAG"]:
    d.mkdir(exist_ok=True, parents=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

with open(CHUNKS_FILE, "r", encoding="utf-8") as fh:
    legal_chunks = json.load(fh)
texts     = [c["text"]     for c in legal_chunks]
chunk_ids = [c["chunk_id"] for c in legal_chunks]
chunk_map = {c["chunk_id"]: c for c in legal_chunks}
print(f"Loaded {len(legal_chunks):,} chunks")

EVALUATION_QUERIES = [
    "Article 226 writ petition",
    "Article 32 fundamental rights",
    "anticipatory bail",
    "forest land",
    "murder conviction",
    "injunction order",
    "basic structure doctrine",
    "double jeopardy",
    "habeas corpus arbitrary detention",
    "public interest litigation environment",
    "freedom of speech and expression",
    "right to privacy data protection",
    "judicial review of administrative action",
    "separation of powers",
    "preventive detention laws",
    "election commission powers",
    "reservation policy and creamy layer",
    "uniform civil code",
    "federalism and state rights",
    "contempt of court proceedings"
]
if LABELS_FILE.exists():
    labels_df = pd.read_csv(LABELS_FILE)
    labels_df["Relevant"] = pd.to_numeric(labels_df["Relevant"], errors="coerce").fillna(0)
else:
    labels_df = pd.DataFrame(columns=["Query", "Chunk_ID", "Relevant"])
print(f"Labels : {labels_df.shape[0]} rows | {len(EVALUATION_QUERIES)} queries")
print(f"  Relevant=1 : {int(labels_df['Relevant'].sum())}")
for i, q in enumerate(EVALUATION_QUERIES, 1):
    print(f"  {i:2d}. {q}")

# Validate chunk count vs existing embeddings
print("\n=== Pre-flight Embedding Check ===")
for model_key in ["MiniLM","InLegalBERT","LegalBERT","BGE"]:
    ep = EXP_DIR / model_key / "embeddings.npy"
    if ep.exists():
        sp = EXP_DIR / model_key / "statistics.json"
        if sp.exists():
            import json as j2
            st = j2.load(open(sp))
            existing_count = st.get("total_chunks", 0)
            match = existing_count == len(legal_chunks)
            status = "OK" if match else f"MISMATCH ({existing_count:,} vs {len(legal_chunks):,}) -> will regenerate"
        else:
            status = "MISMATCH (statistics.json missing) -> will regenerate"
        print(f"  [{model_key}] {status}")
    else:
        print(f"  [{model_key}] Not found -> will generate")

Device : cuda
PyTorch: 2.4.1+cu118
Loaded 6,597 chunks
Labels : 458 rows | 20 queries
  Relevant=1 : 326
   1. Article 226 writ petition
   2. Article 32 fundamental rights
   3. anticipatory bail
   4. forest land
   5. murder conviction
   6. injunction order
   7. basic structure doctrine
   8. double jeopardy
   9. habeas corpus arbitrary detention
  10. public interest litigation environment
  11. freedom of speech and expression
  12. right to privacy data protection
  13. judicial review of administrative action
  14. separation of powers
  15. preventive detention laws
  16. election commission powers
  17. reservation policy and creamy layer
  18. uniform civil code
  19. federalism and state rights
  20. contempt of court proceedings

=== Pre-flight Embedding Check ===
  [MiniLM] OK
  [InLegalBERT] OK
  [LegalBERT] OK
  [BGE] OK


---
## Section 2 - Embedding Engines & FAISS Indexing
Loads existing indices for MiniLM, InLegalBERT, BGE. Generates LegalBERT if missing.

In [2]:
# ============================================================
# SECTION 2 -- EMBEDDING ENGINES & FAISS INDEXING
# ============================================================
from transformers import AutoTokenizer, AutoModel

MODELS = {
    "MiniLM"      : "sentence-transformers/all-MiniLM-L6-v2",
    "InLegalBERT" : "law-ai/InLegalBERT",
    "LegalBERT"   : "nlpaueb/legal-bert-base-uncased",
    "BGE"         : "BAAI/bge-base-en-v1.5",
}
MODEL_DIRS = {k: EXP_DIR / k for k in MODELS}
for d in MODEL_DIRS.values():
    d.mkdir(exist_ok=True, parents=True)

def mean_pooling(model_output, attention_mask):
    tok  = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(tok.size()).float()
    return torch.sum(tok * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

def embed_texts(model_name, texts_list, batch_size=32, max_len=512):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(texts_list), batch_size), desc=f"Embed {model_name.split('/')[-1]}"):
        batch = texts_list[i:i+batch_size]
        enc   = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        emb = mean_pooling(out, enc["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_emb.append(emb.cpu().numpy())
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.vstack(all_emb).astype("float32")

faiss_indices  = {}
all_embeddings = {}
model_stats    = {}

for mkey, mname in MODELS.items():
    mdir      = MODEL_DIRS[mkey]
    emb_path  = mdir / "embeddings.npy"
    idx_path  = mdir / "index.faiss"
    stat_path = mdir / "statistics.json"

    # Force regeneration if chunk count doesn't match current corpus
    needs_regen = True
    if emb_path.exists() and idx_path.exists() and stat_path.exists():
        import json as j2
        st = j2.load(open(stat_path))
        existing_count = st.get("total_chunks", 0)
        if existing_count == len(texts):
            print(f"[{mkey}] Loading existing index (chunks match: {len(texts):,}) ...")
            emb   = np.load(str(emb_path))
            index = faiss.read_index(str(idx_path))
            needs_regen = False
        else:
            print(f"[{mkey}] Chunk count mismatch! Existing={existing_count:,}, Current={len(texts):,} -> REGENERATING ...")
    if needs_regen:
        print(f"[{mkey}] Generating embeddings ({mname}) ...")
        t0  = time.time()
        emb = embed_texts(mname, texts)
        elapsed_emb = time.time() - t0
        np.save(str(emb_path), emb)
        dim   = emb.shape[1]
        t1    = time.time()
        index = faiss.IndexFlatIP(dim)
        index.add(emb)
        elapsed_idx = time.time() - t1
        faiss.write_index(index, str(idx_path))
        st = {
            "model": mkey, "model_name": mname,
            "embedding_dimension": int(dim), "total_chunks": len(texts),
            "embedding_time_seconds": round(elapsed_emb, 2),
            "index_build_time_seconds": round(elapsed_idx, 4),
            "average_query_time_ms": 0.0,
            "minimum_similarity": round(float(emb.min()), 4),
            "maximum_similarity": round(float(emb.max()), 4),
            "average_similarity": round(float(emb.mean()), 4),
        }
        with open(stat_path, "w") as f:
            json.dump(st, f, indent=2)

    faiss_indices[mkey]  = index
    all_embeddings[mkey] = emb
    if stat_path.exists():
        with open(stat_path) as f:
            model_stats[mkey] = json.load(f)
    print(f"  [{mkey}] dim={emb.shape[1]}  chunks={emb.shape[0]:,}")

print("All models ready.\n")

eff_rows = []
for mkey, st in model_stats.items():
    eff_rows.append({
        "Model": mkey, "HuggingFace ID": st.get("model_name",""),
        "Dimension": st.get("embedding_dimension","?"),
        "Embed Time (s)": st.get("embedding_time_seconds","?"),
        "Avg Query (ms)": st.get("average_query_time_ms","?"),
        "Avg Sim": round(st.get("average_similarity",0), 4),
    })
print("=== Model Efficiency Summary ===")
display(pd.DataFrame(eff_rows))

[MiniLM] Loading existing index (chunks match: 6,597) ...
  [MiniLM] dim=384  chunks=6,597
[InLegalBERT] Loading existing index (chunks match: 6,597) ...
  [InLegalBERT] dim=768  chunks=6,597
[LegalBERT] Loading existing index (chunks match: 6,597) ...
  [LegalBERT] dim=768  chunks=6,597
[BGE] Loading existing index (chunks match: 6,597) ...
  [BGE] dim=768  chunks=6,597
All models ready.

=== Model Efficiency Summary ===


,Model,HuggingFace ID,Dimension,Embed Time (s),Avg Query (ms),Avg Sim
0,MiniLM,sentence-transformers/all-MiniLM-L6-v2,384,55.75,0.0,-0.0004
1,InLegalBERT,law-ai/InLegalBERT,768,318.20,0.0,-0.0019
2,LegalBERT,nlpaueb/legal-bert-base-uncased,768,314.59,0.0,-0.0015
3,BGE,BAAI/bge-base-en-v1.5,768,298.17,0.0,-0.0007


---
## Section 3 - Citation Graph Construction (GraphRAG)
Builds bipartite citation/act graph. Computes PageRank per chunk and Jaccard citation similarity.

In [3]:
# ============================================================
# SECTION 3 -- CITATION GRAPH (GraphRAG)
# ============================================================
def parse_citations(raw) -> set:
    if not raw or (isinstance(raw, float) and pd.isna(raw)):
        return set()
    return {c.strip() for c in str(raw).split(";") if c.strip()}

chunk_citations = {
    c["chunk_id"]: parse_citations(c.get("citations","")) | parse_citations(c.get("acts",""))
    for c in legal_chunks
}
chunk_ref_sets = {cid: refs for cid, refs in chunk_citations.items() if refs}

print("Building citation bipartite graph ...")
G = nx.Graph()
chunk_nodes, ref_nodes = set(), set()
for cid, refs in chunk_citations.items():
    G.add_node(cid, bipartite=0)
    chunk_nodes.add(cid)
    for ref in refs:
        rn = "REF::" + ref
        G.add_node(rn, bipartite=1)
        ref_nodes.add(rn)
        G.add_edge(cid, rn)
print(f"  Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")
print(f"  Chunk nodes: {len(chunk_nodes):,}  Reference nodes: {len(ref_nodes):,}")

print("Computing PageRank ...")
pagerank_raw  = nx.pagerank(G, alpha=0.85, max_iter=200, tol=1e-6)
chunk_pr      = {cid: pagerank_raw.get(cid, 0) for cid in chunk_ids}
pr_vals       = np.array(list(chunk_pr.values()))
pr_min, pr_max = pr_vals.min(), pr_vals.max()
chunk_pr_norm = {cid: (v - pr_min) / (pr_max - pr_min + 1e-12) for cid, v in chunk_pr.items()}

def jaccard_sim(a: set, b: set) -> float:
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

def graph_score(chunk_id: str, query_ref_context: set = None) -> float:
    pr = chunk_pr_norm.get(chunk_id, 0.0)
    j  = jaccard_sim(chunk_ref_sets.get(chunk_id, set()), query_ref_context) if query_ref_context else 0.0
    return 0.7 * pr + 0.3 * j

print(f"PageRank stats: min={pr_vals.min():.6f}  max={pr_vals.max():.6f}  mean={pr_vals.mean():.6f}")
top_pr = sorted(chunk_pr.items(), key=lambda x: x[1], reverse=True)[:10]
top_pr_df = pd.DataFrame(top_pr, columns=["Chunk_ID","PageRank"])
top_pr_df["Case_Title"] = top_pr_df["Chunk_ID"].apply(lambda x: chunk_map.get(x,{}).get("case_title","")[:55])
print("\nTop-10 chunks by PageRank:")
display(top_pr_df)

pr_df = pd.DataFrame([{"Chunk_ID":cid,"PageRank":chunk_pr[cid],"PageRank_Norm":chunk_pr_norm[cid]} for cid in chunk_ids])
pr_df.to_csv(EXP_DIR / "chunk_pagerank.csv", index=False)
print(f"Saved -> {EXP_DIR / 'chunk_pagerank.csv'}")

# PageRank distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(pr_vals, bins=80, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].set_title("PageRank Distribution (all chunks)", fontsize=13)
axes[0].set_xlabel("PageRank Score"); axes[0].set_ylabel("Count"); axes[0].set_yscale("log")
top30  = sorted(chunk_pr.items(), key=lambda x: x[1], reverse=True)[:30]
t_ids  = [x[0] for x in top30]; t_vals = [x[1] for x in top30]
axes[1].barh(range(30), t_vals[::-1], color="#DD8452", edgecolor="white")
axes[1].set_yticks(range(30))
axes[1].set_yticklabels([x[:30] for x in t_ids[::-1]], fontsize=7)
axes[1].set_title("Top-30 Chunks by PageRank", fontsize=13)
axes[1].set_xlabel("PageRank Score")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_pagerank_distribution.png", dpi=150)
plt.show()
print("Saved -> figures/fig_pagerank_distribution.png")

Building citation bipartite graph ...


  Nodes: 8,253  Edges: 261,193
  Chunk nodes: 6,597  Reference nodes: 1,656
Computing PageRank ...


PageRank stats: min=0.000018  max=0.000156  mean=0.000079

Top-10 chunks by PageRank:


,Chunk_ID,PageRank,Case_Title
0,CASE0043_CHUNK_0001,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
1,CASE0043_CHUNK_0002,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
2,CASE0043_CHUNK_0003,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
3,CASE0043_CHUNK_0004,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
4,CASE0043_CHUNK_0005,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
5,CASE0043_CHUNK_0006,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
6,CASE0043_CHUNK_0007,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
7,CASE0043_CHUNK_0008,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
8,CASE0043_CHUNK_0009,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...
9,CASE0043_CHUNK_0010,0.000156,Kaushal Kishor vs The State Of Uttar Pradesh G...


Saved -> E:\Legal Finale\data\experiments\chunk_pagerank.csv


Saved -> figures/fig_pagerank_distribution.png


---
## Section 4 - Retrieval Pipelines
Dense (x4), BM25, GraphRAG-Augmented (x3), RRF-All Ensemble.

In [4]:
# ============================================================
# SECTION 4 -- RETRIEVAL PIPELINES
# ============================================================
_loaded_models = {}

def get_model(mkey):
    if mkey not in _loaded_models:
        mname = MODELS[mkey]
        tok   = AutoTokenizer.from_pretrained(mname)
        mdl   = AutoModel.from_pretrained(mname).to(DEVICE)
        mdl.eval()
        _loaded_models[mkey] = (tok, mdl)
    return _loaded_models[mkey]

def encode_query(mkey, query, max_len=512):
    tok, mdl = get_model(mkey)
    enc = tok([query], padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = mdl(**enc)
    emb = mean_pooling(out, enc["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy().astype("float32")

def search_dense(query, model_key, top_k=5):
    qemb  = encode_query(model_key, query)
    scores, idxs = faiss_indices[model_key].search(qemb, top_k)
    results = []
    for rank, (score, i) in enumerate(zip(scores[0], idxs[0]), 1):
        if i < 0 or i >= len(legal_chunks): continue
        c = legal_chunks[i]
        results.append({
            "Query": query, "Retriever": model_key,
            "Rank": rank, "Cosine_Score": float(score),
            "Graph_Score": np.nan, "Combined_Score": np.nan,
            "Chunk_ID": c["chunk_id"], "Case_ID": c["case_id"],
            "Court": c["court"], "Case_Title": c["case_title"],
            "Chunk_Text": c["text"],
        })
    return results

def search_graphrag(query, model_key, top_k=5, alpha=0.3, beta=0.7, candidate_k=100):
    qemb  = encode_query(model_key, query)
    scores, idxs = faiss_indices[model_key].search(qemb, candidate_k)
    
    # Extract top-10 hits for Personalized PageRank seed
    seed_nodes = {}
    for i in idxs[0][:10]:
        if 0 <= i < len(legal_chunks):
            cid = legal_chunks[i]["chunk_id"]
            if cid in G.nodes():
                seed_nodes[cid] = 1.0
                
    if seed_nodes:
        total = sum(seed_nodes.values())
        seed_nodes = {k: v/total for k, v in seed_nodes.items()}
        try:
            ppr = nx.pagerank(G, alpha=0.85, personalization=seed_nodes, max_iter=100, tol=1e-5)
        except:
            ppr = chunk_pr # fallback
    else:
        ppr = chunk_pr # fallback
        
    cand_ppr = [ppr.get(legal_chunks[i]["chunk_id"], 0.0) for i in idxs[0] if 0 <= i < len(legal_chunks)]
    min_p, max_p = (min(cand_ppr), max(cand_ppr)) if cand_ppr else (0, 1)
    
    candidates = []
    for raw_score, i in zip(scores[0], idxs[0]):
        if i < 0 or i >= len(legal_chunks): continue
        c   = legal_chunks[i]; cid = c["chunk_id"]
        
        p_score = ppr.get(cid, 0.0)
        norm_p = (p_score - min_p) / (max_p - min_p + 1e-12) if max_p > min_p else 0.0
        combined = alpha * float(raw_score) + beta * norm_p
        
        candidates.append({
            "Query": query, "Retriever": f"GraphRAG-{model_key}",
            "Cosine_Score": float(raw_score), "Graph_Score": norm_p, "Combined_Score": combined,
            "Chunk_ID": cid, "Case_ID": c["case_id"],
            "Court": c["court"], "Case_Title": c["case_title"], "Chunk_Text": c["text"],
        })
    candidates.sort(key=lambda x: x["Combined_Score"], reverse=True)
    for rank, item in enumerate(candidates[:top_k], 1):
        item["Rank"] = rank
    return candidates[:top_k]

# Sanity check
_q = EVALUATION_QUERIES[0] if EVALUATION_QUERIES else "Article 226 writ petition"
print(f"Sanity check -- '{_q}'")
for r in search_dense(_q, "BGE", top_k=2):
    print(f"  [BGE] rank={r['Rank']} cos={r['Cosine_Score']:.4f} | {r['Chunk_ID']}")
for r in search_graphrag(_q, "BGE", top_k=2):
    print(f"  [GraphRAG-BGE] rank={r['Rank']} cos={r['Cosine_Score']:.4f} gs={r['Graph_Score']:.4f} comb={r['Combined_Score']:.4f}")
print("All retrievers ready.")

Sanity check -- 'Article 226 writ petition'


  [BGE] rank=1 cos=0.8003 | CASE0142_CHUNK_0044
  [BGE] rank=2 cos=0.7879 | CASE0020_CHUNK_0008


  [GraphRAG-BGE] rank=1 cos=0.7819 gs=1.0000 comb=0.9346
  [GraphRAG-BGE] rank=2 cos=0.7781 gs=0.9990 comb=0.9327
All retrievers ready.


---
## Section 5 - Run All Retrievals
9 retrievers x 20 queries (top-5). Saves combined CSV.

In [5]:
# ============================================================
# SECTION 5 -- RUN ALL RETRIEVALS
# ============================================================
TOP_K = 5
RETRIEVERS = (
    list(MODELS.keys())
    + [f"GraphRAG-{m}" for m in MODELS.keys()]
)
print(f"Running {len(RETRIEVERS)} retrievers x {len(EVALUATION_QUERIES)} queries (top-{TOP_K}) ...")

all_results = []
for q in tqdm(EVALUATION_QUERIES, desc="Queries"):
    for mkey in MODELS.keys():
        all_results.extend(search_dense(q, mkey, top_k=TOP_K))
        all_results.extend(search_graphrag(q, mkey, top_k=TOP_K))

raw_df = pd.DataFrame(all_results)
raw_df.to_csv(EXP_DIR / "all_model_results_v2.csv", index=False)
print(f"Saved {len(raw_df):,} rows -> {EXP_DIR / 'all_model_results_v2.csv'}")
print("\nRetriever row counts:")
display(raw_df.groupby("Retriever").size().reset_index(name="Rows"))

Running 8 retrievers x 20 queries (top-5) ...



Queries:   0%|          | 0/20 [00:00<?, ?it/s]


Queries:   5%|▌         | 1/20 [00:06<02:05,  6.63s/it]


Queries:  10%|█         | 2/20 [00:08<01:10,  3.91s/it]


Queries:  15%|█▌        | 3/20 [00:10<00:51,  3.05s/it]

...


Queries:  95%|█████████▌| 19/20 [00:43<00:02,  2.02s/it]


Queries: 100%|██████████| 20/20 [00:45<00:00,  2.02s/it]


Queries: 100%|██████████| 20/20 [00:45<00:00,  2.26s/it]

Saved 800 rows -> E:\Legal Finale\data\experiments\all_model_results_v2.csv

Retriever row counts:


,Retriever,Rows
0,BGE,100
1,GraphRAG-BGE,100
2,GraphRAG-InLegalBERT,100
3,GraphRAG-LegalBERT,100
4,GraphRAG-MiniLM,100
5,InLegalBERT,100
6,LegalBERT,100
7,MiniLM,100


---
## Section 6 - Relevance Label Merging, Coverage & Labeler

**Important**: Run `data/evaluation/relevance_labeler.py` to label additional queries.
The labeler reads `master_evaluation_sheet.csv` and writes `completed_labels.csv`.

> **Missing chunks**: A small number of chunk IDs in labels may not exist in the
> current corpus (chunks were re-processed). These are handled gracefully below.

In [6]:
# ============================================================
# SECTION 6 -- RELEVANCE LABEL MERGING & COVERAGE
# ============================================================
from IPython.display import display

# --- 6a. Merge eval results with relevance labels ---
eval_df = pd.merge(
    raw_df,
    labels_df[["Query","Chunk_ID","Relevant"]],
    on=["Query","Chunk_ID"], how="left"
)
eval_df["Relevant"] = pd.to_numeric(eval_df["Relevant"], errors="coerce").fillna(0).astype(int)

# --- 6b. Diagnose any orphan chunk IDs in labels ---
current_chunk_ids = set(chunk_ids)
label_chunk_ids   = set(labels_df["Chunk_ID"].unique())
orphan_ids        = label_chunk_ids - current_chunk_ids
if orphan_ids:
    print(f"WARNING: {len(orphan_ids)} chunk IDs in labels are NOT in current corpus:")
    for cid in sorted(orphan_ids):
        rows = labels_df[labels_df["Chunk_ID"]==cid]
        for _, r in rows.iterrows():
            print(f"  {cid}  rel={r['Relevant']}  query={r['Query']!r}")
    print("  -> These will NOT be counted as retrievable (not in FAISS index).")
else:
    print("OK: All labelled chunk IDs exist in current corpus.")

# --- 6c. Compute total relevant per query (from labels only, present chunks) ---
valid_labels   = labels_df[labels_df["Chunk_ID"].isin(current_chunk_ids)].copy()
valid_labels["Relevant"] = pd.to_numeric(valid_labels["Relevant"], errors="coerce").fillna(0)
total_relevant = valid_labels.groupby("Query")["Relevant"].sum().to_dict()

print("\nGround-truth relevant chunk count per query (current corpus only):")
for q, n in sorted(total_relevant.items()):
    print(f"  {int(n):2d}  {q}")

# --- 6d. Regenerate master_evaluation_sheet.csv from current retrieval results ---
eval_sheet = raw_df[["Query","Chunk_ID","Retriever","Rank","Cosine_Score"]].copy()
eval_sheet = eval_sheet.sort_values(["Query","Retriever","Rank"])
# Merge in chunk metadata
for col, key in [("Case_ID","case_id"),("Court","court"),("Case_Title","case_title"),("Chunk_Text","text")]:
    eval_sheet[col] = eval_sheet["Chunk_ID"].apply(lambda x: chunk_map.get(x,{}).get(key,""))
# Merge in existing labels (to preserve already-labelled data)
eval_sheet = pd.merge(eval_sheet,
                      labels_df[["Query","Chunk_ID","Relevant"]],
                      on=["Query","Chunk_ID"], how="left")
# Only keep unique Query+Chunk_ID rows for the labeler
labeler_df = eval_sheet.drop_duplicates(subset=["Query","Chunk_ID"])[
    ["Query","Case_ID","Chunk_ID","Court","Case_Title","Chunk_Text","Relevant"]
].reset_index(drop=True)
labeler_df.to_csv(EXP_DIR/"master_evaluation_sheet.csv", index=False)
print(f"\nmaster_evaluation_sheet.csv updated: {len(labeler_df)} rows, {labeler_df['Query'].nunique()} queries")
print(f"  Already labelled: {labeler_df['Relevant'].notna().sum()}")
print(f"  Needs labelling : {labeler_df['Relevant'].isna().sum()}")

# --- 6e. Coverage heatmap (robust - no truncation issues) ---
coverage = []
for q in EVALUATION_QUERIES:
    row = {"Query": q[:45]}
    for ret in RETRIEVERS:
        sub = eval_df[(eval_df["Query"]==q) & (eval_df["Retriever"]==ret)]
        found = int(sub["Relevant"].sum() > 0)
        row[ret] = found
    coverage.append(row)
cov_df = pd.DataFrame(coverage).set_index("Query")

fig, ax = plt.subplots(figsize=(20, max(6, len(EVALUATION_QUERIES)*0.45)))
sns.heatmap(cov_df.astype(float), annot=True, fmt=".0f", cmap="YlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Found >=1 relevant"})
ax.set_title("Coverage Heatmap: Did Retriever Find >=1 Relevant Chunk per Query?", fontsize=13)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9, rotation=0)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_coverage_heatmap.png", dpi=150)
plt.show()
print("Saved -> figures/fig_coverage_heatmap.png")

# --- 6f. Per-retriever coverage summary ---
cov_summary = cov_df.sum(axis=0).reset_index()
cov_summary.columns = ["Retriever","Queries_With_Hit"]
cov_summary["Coverage_%"] = (100 * cov_summary["Queries_With_Hit"] / len(EVALUATION_QUERIES)).round(1)
cov_summary = cov_summary.sort_values("Coverage_%", ascending=False)
print("\nPer-retriever query coverage:")
display(cov_summary)

# --- 6g. Relevant hits table ---
hits_df = eval_df.groupby(["Retriever","Query"])["Relevant"].sum().unstack(fill_value=0)
print("\nRelevant hits per (Retriever, Query):")
display(hits_df)

# --- 6h. Labeler launcher helper ---
print("\n--- To label more chunks manually, run: ---")
print("  python data/evaluation/relevance_labeler.py")
print("  (Keyboard shortcuts: y=Relevant, n=Not Relevant, s=Skip)")

OK: All labelled chunk IDs exist in current corpus.

Ground-truth relevant chunk count per query (current corpus only):
  13  Article 226 writ petition
  17  Article 32 fundamental rights
  15  anticipatory bail
  19  basic structure doctrine
  17  contempt of court proceedings
  19  double jeopardy
  14  election commission powers
  17  federalism and state rights
  17  forest land
  10  freedom of speech and expression
  16  habeas corpus arbitrary detention
  16  injunction order
  14  judicial review of administrative action
  20  murder conviction
  15  preventive detention laws
  17  public interest litigation environment
  18  reservation policy and creamy layer
  16  right to privacy data protection
  18  separation of powers
  18  uniform civil code

master_evaluation_sheet.csv updated: 458 rows, 20 queries
  Already labelled: 458
  Needs labelling : 0


Saved -> figures/fig_coverage_heatmap.png

Per-retriever query coverage:


,Retriever,Queries_With_Hit,Coverage_%
0,MiniLM,20,100.0
1,InLegalBERT,20,100.0
2,LegalBERT,20,100.0
3,BGE,20,100.0
4,GraphRAG-MiniLM,20,100.0
5,GraphRAG-InLegalBERT,20,100.0
6,GraphRAG-LegalBERT,20,100.0
7,GraphRAG-BGE,20,100.0



Relevant hits per (Retriever, Query):


Query,Article 226 writ petition,Article 32 fundamental rights,anticipatory bail,basic structure doctrine,contempt of court proceedings,double jeopardy,election commission powers,federalism and state rights,forest land,freedom of speech and expression,habeas corpus arbitrary detention,injunction order,judicial review of administrative action,murder conviction,preventive detention laws,public interest litigation environment,reservation policy and creamy layer,right to privacy data protection,separation of powers,uniform civil code
Retriever,,,,,,,,,,,,,,,,,,,,
BGE,3,3,5,4,3,5,4,2,4,4,5,4,3,4,5,5,3,3,3,5
GraphRAG-BGE,5,5,5,4,5,5,5,5,4,4,5,5,4,5,5,5,5,5,5,5
GraphRAG-InLegalBERT,4,4,5,5,3,5,4,5,5,5,3,5,4,5,4,4,5,4,4,5
GraphRAG-LegalBERT,3,5,5,5,5,4,3,4,5,4,4,3,5,4,5,4,4,4,5,3
GraphRAG-MiniLM,4,4,5,5,5,4,5,5,4,3,5,5,4,4,5,4,5,5,4,4
InLegalBERT,3,4,3,3,3,4,1,4,2,5,2,4,4,4,4,2,3,2,4,4
LegalBERT,3,4,3,3,1,3,2,2,3,4,3,3,4,4,4,2,3,3,2,3
MiniLM,4,3,5,4,3,3,5,5,3,3,5,4,4,3,4,3,5,3,3,3



--- To label more chunks manually, run: ---
  python data/evaluation/relevance_labeler.py
  (Keyboard shortcuts: y=Relevant, n=Not Relevant, s=Skip)


---
## Section 7 - Core IR Metrics
Precision@k, Recall@k, F1@k, MRR, MAP, NDCG@k for all retrievers.

In [7]:
# ============================================================
# SECTION 7 -- CORE IR METRICS
# ============================================================
def precision_at_k(rels, k): return sum(rels[:k]) / k
def recall_at_k(rels, tot, k): return sum(rels[:k]) / tot if tot > 0 else 0.0
def f1_at_k(rels, tot, k):
    p, r = precision_at_k(rels, k), recall_at_k(rels, tot, k)
    return 2*p*r/(p+r) if (p+r)>0 else 0.0
def avg_precision(rels, tot):
    if tot==0: return 0.0
    hits, ap = 0, 0.0
    for i, r in enumerate(rels, 1):
        if r==1: hits+=1; ap+=hits/i
    return ap/tot
def mrr(rels):
    for i, r in enumerate(rels, 1):
        if r==1: return 1.0/i
    return 0.0
def ndcg_at_k(rels, k):
    dcg  = sum((2**r-1)/math.log2(i+1) for i,r in enumerate(rels[:k],1))
    idcg = sum((2**r-1)/math.log2(i+1) for i,r in enumerate(sorted(rels,reverse=True)[:k],1))
    return dcg/idcg if idcg>0 else 0.0

metrics_rows   = []
per_query_ndcg = defaultdict(dict)

for ret in RETRIEVERS:
    ret_df  = eval_df[eval_df["Retriever"]==ret]
    agg     = {k:[] for k in ["P@1","P@3","P@5","R@3","R@5","F1@3","MRR","MAP","NDCG@3","NDCG@5","NDCG@10"]}
    q_count = 0
    for q in EVALUATION_QUERIES:
        qdf = ret_df[ret_df["Query"]==q].sort_values("Rank")
        if len(qdf)==0: continue
        q_count += 1
        rels    = qdf["Relevant"].tolist()
        tot_rel = total_relevant.get(q, 0)
        agg["P@1"].append(precision_at_k(rels,1))
        agg["P@3"].append(precision_at_k(rels,3))
        agg["P@5"].append(precision_at_k(rels,5))
        agg["R@3"].append(recall_at_k(rels,tot_rel,3))
        agg["R@5"].append(recall_at_k(rels,tot_rel,5))
        agg["F1@3"].append(f1_at_k(rels,tot_rel,3))
        agg["MRR"].append(mrr(rels))
        agg["MAP"].append(avg_precision(rels,tot_rel))
        agg["NDCG@3"].append(ndcg_at_k(rels,3))
        ndcg5 = ndcg_at_k(rels,5)
        agg["NDCG@5"].append(ndcg5)
        agg["NDCG@10"].append(ndcg_at_k(rels,10))
        per_query_ndcg[ret][q] = ndcg5
    if q_count>0:
        metrics_rows.append({
            "Retriever": ret,
            **{k: round(np.mean(v),4) for k,v in agg.items()},
            "Queries_Evaluated": q_count,
        })

metrics_df = pd.DataFrame(metrics_rows).sort_values("NDCG@5", ascending=False)
metrics_df.to_csv(EXP_DIR/"full_metrics_v2.csv", index=False)
print("=== FULL IR METRICS (sorted by NDCG@5) ===")
display(metrics_df)
print(f"\nSaved -> {EXP_DIR/'full_metrics_v2.csv'}")

=== FULL IR METRICS (sorted by NDCG@5) ===


,Retriever,P@1,P@3,P@5,R@3,R@5,F1@3,MRR,MAP,NDCG@3,NDCG@5,NDCG@10,Queries_Evaluated
7,GraphRAG-BGE,1.00,0.9667,0.96,0.1806,0.3000,0.3033,1.0000,0.2948,0.9735,0.9922,0.9922,20
5,GraphRAG-InLegalBERT,0.80,0.9167,0.88,0.1728,0.2759,0.2895,0.9000,0.2564,0.8944,0.9447,0.9447,20
4,GraphRAG-MiniLM,0.80,0.8667,0.89,0.1640,0.2772,0.2749,0.9000,0.2536,0.8531,0.9359,0.9359,20
3,BGE,0.90,0.7667,0.77,0.1436,0.2416,0.2411,0.9417,0.2144,0.8056,0.9324,0.9324,20
6,GraphRAG-LegalBERT,0.80,0.8167,0.84,0.1544,0.2625,0.2587,0.8917,0.2381,0.8148,0.9259,0.9259,20
0,MiniLM,0.80,0.8000,0.75,0.1514,0.2365,0.2537,0.9000,0.2100,0.8092,0.9246,0.9246,20
1,InLegalBERT,0.60,0.7167,0.65,0.1348,0.2058,0.2259,0.8000,0.1698,0.7521,0.8671,0.8671,20
2,LegalBERT,0.55,0.6000,0.59,0.1150,0.1877,0.1920,0.7625,0.1449,0.6305,0.8239,0.8239,20



Saved -> E:\Legal Finale\data\experiments\full_metrics_v2.csv


---
## Section 8 - Cosine Similarity Distribution Analysis
Per-model score histograms, violin plots, relevant vs non-relevant comparisons.

In [8]:
# ============================================================
# SECTION 8 -- COSINE SIMILARITY DISTRIBUTIONS
# ============================================================
DENSE_RETS = list(MODELS.keys())
COLORS     = ["#4C72B0","#DD8452","#55A868","#C44E52"]
sim_data   = {}
sim_stats  = []

for ret in DENSE_RETS:
    scores = eval_df[eval_df["Retriever"]==ret]["Cosine_Score"].dropna().values
    sim_data[ret] = scores
    sim_stats.append({
        "Retriever": ret, "N": len(scores),
        "Mean": round(scores.mean(),4), "Std": round(scores.std(),4),
        "Min":  round(scores.min(),4),
        "Q25":  round(np.percentile(scores,25),4),
        "Median": round(np.median(scores),4),
        "Q75":  round(np.percentile(scores,75),4),
        "Max":  round(scores.max(),4),
        "IQR":  round(np.percentile(scores,75)-np.percentile(scores,25),4),
    })
sim_stats_df = pd.DataFrame(sim_stats)
print("=== Cosine Similarity Statistics ===")
display(sim_stats_df)
sim_stats_df.to_csv(EXP_DIR/"cosine_similarity_stats.csv", index=False)

# Fig 1: Histograms per model
fig, axes = plt.subplots(2,2,figsize=(14,10))
for ax, ret, col in zip(axes.flat, DENSE_RETS, COLORS):
    data = sim_data[ret]
    ax.hist(data, bins=30, color=col, edgecolor="white", alpha=0.85, density=True)
    ax.axvline(data.mean(),     color="black", ls="--", lw=1.5, label=f"mean={data.mean():.3f}")
    ax.axvline(np.median(data), color="gray",  ls=":",  lw=1.5, label=f"med={np.median(data):.3f}")
    ax.set_title(f"{ret} -- Cosine Score Distribution", fontsize=12)
    ax.set_xlabel("Cosine Similarity"); ax.set_ylabel("Density")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.suptitle("Cosine Similarity Distributions (top-5 per query)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_cosine_histograms.png", dpi=150)
plt.show()
print("Saved -> figures/fig_cosine_histograms.png")

# Fig 2: Violin plot
fig, ax = plt.subplots(figsize=(10,6))
parts = ax.violinplot([sim_data[r] for r in DENSE_RETS],
                      positions=range(len(DENSE_RETS)), showmedians=True, showextrema=True)
for body, col in zip(parts["bodies"], COLORS):
    body.set_facecolor(col); body.set_alpha(0.7)
ax.set_xticks(range(len(DENSE_RETS))); ax.set_xticklabels(DENSE_RETS, fontsize=11)
ax.set_ylabel("Cosine Similarity Score", fontsize=11)
ax.set_title("Cosine Similarity -- Violin Plot (All Dense Models)", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_cosine_violin.png", dpi=150)
plt.show()
print("Saved -> figures/fig_cosine_violin.png")

# Fig 3: Relevant vs Non-Relevant cosine scores
from scipy.stats import mannwhitneyu
fig, axes = plt.subplots(2,2,figsize=(14,10))
for ax, ret, col in zip(axes.flat, DENSE_RETS, COLORS):
    sub = eval_df[eval_df["Retriever"]==ret][["Cosine_Score","Relevant"]].dropna()
    rel = sub[sub["Relevant"]==1]["Cosine_Score"].values
    nr  = sub[sub["Relevant"]==0]["Cosine_Score"].values
    bins = np.linspace(sub["Cosine_Score"].min(), sub["Cosine_Score"].max(), 25)
    ax.hist(nr,  bins=bins, color="#aec7e8", label="Non-Relevant", alpha=0.75, density=True)
    ax.hist(rel, bins=bins, color=col,       label="Relevant",     alpha=0.75, density=True)
    title_str = ret
    if len(rel)>0 and len(nr)>0:
        try:
            t, p = mannwhitneyu(rel, nr, alternative="greater")
            title_str += f"  (MWU p={p:.3f})"
        except Exception:
            pass
    ax.set_title(title_str, fontsize=11)
    ax.set_xlabel("Cosine Score"); ax.set_ylabel("Density")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.suptitle("Cosine Score: Relevant vs Non-Relevant Chunks", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_cosine_rel_vs_nonrel.png", dpi=150)
plt.show()
print("Saved -> figures/fig_cosine_rel_vs_nonrel.png")

=== Cosine Similarity Statistics ===


,Retriever,N,Mean,Std,Min,Q25,Median,Q75,Max,IQR
0,MiniLM,100,0.5194,0.1028,0.3050,0.4641,0.5347,0.5915,0.7382,0.1275
1,InLegalBERT,100,0.5185,0.1148,0.2244,0.4831,0.5431,0.5927,0.6755,0.1096
2,LegalBERT,100,0.5573,0.1435,0.2413,0.4705,0.5821,0.6317,0.8171,0.1613
3,BGE,100,0.7514,0.0351,0.6733,0.7245,0.7531,0.7790,0.8272,0.0545


Saved -> figures/fig_cosine_histograms.png
Saved -> figures/fig_cosine_violin.png


Saved -> figures/fig_cosine_rel_vs_nonrel.png


---
## Section 9 - Graph Score Analysis
FAISS vs Graph score scatter, alpha-ablation on NDCG@5, PageRank: relevant vs non-relevant.

In [9]:
# ============================================================
# SECTION 9 -- GRAPH SCORE ANALYSIS
# ============================================================
graphrag_df = eval_df[eval_df["Retriever"].str.startswith("GraphRAG")].dropna(subset=["Graph_Score"]).copy()

if graphrag_df.empty:
    print("No GraphRAG results -- skipping graph score analysis.")
else:
    g_models = graphrag_df["Retriever"].unique().tolist()
    fig, axes = plt.subplots(1, len(g_models), figsize=(6*len(g_models), 5))
    if len(g_models)==1: axes=[axes]
    for ax, gret, col in zip(axes, g_models, COLORS):
        sub = graphrag_df[graphrag_df["Retriever"]==gret]
        rel = sub["Relevant"].values
        ax.scatter(sub["Cosine_Score"], sub["Graph_Score"],
                   c=["#e55039" if r==1 else "#95a5a6" for r in rel],
                   alpha=0.65, s=40, edgecolors="none")
        if len(sub)>2:
            r_corr = sub["Cosine_Score"].corr(sub["Graph_Score"])
            z = np.polyfit(sub["Cosine_Score"], sub["Graph_Score"], 1)
            xs = np.linspace(sub["Cosine_Score"].min(), sub["Cosine_Score"].max(), 100)
            ax.plot(xs, np.poly1d(z)(xs), "k--", lw=1.2, label=f"r={r_corr:.2f}")
        ax.set_xlabel("FAISS Cosine Score"); ax.set_ylabel("Graph Score")
        ax.set_title(gret, fontsize=11)
        ax.legend(handles=[
            mpatches.Patch(color="#e55039", label="Relevant"),
            mpatches.Patch(color="#95a5a6", label="Non-Relevant"),
        ], fontsize=8)
        ax.grid(alpha=0.3)
    plt.suptitle("FAISS Cosine vs Graph Score (GraphRAG Retrievers)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGS_DIR/"fig_faiss_vs_graph_score.png", dpi=150)
    plt.show()
    print("Saved -> figures/fig_faiss_vs_graph_score.png")

    # Alpha ablation study
    print("\nRunning alpha-ablation on GraphRAG-BGE ...")
    alpha_vals = [round(x*0.1,1) for x in range(1,11)]
    ablation   = []
    for alpha in tqdm(alpha_vals, desc="alpha"):
        beta   = round(1.0-alpha, 1)
        ndcg5s = []
        for q in EVALUATION_QUERIES:
            res  = search_graphrag(q, "BGE", top_k=5, alpha=alpha, beta=beta)
            rels = []
            for item in sorted(res, key=lambda x: x["Rank"]):
                lrow = labels_df[(labels_df["Query"]==q)&(labels_df["Chunk_ID"]==item["Chunk_ID"])]
                rels.append(int(lrow["Relevant"].values[0]) if len(lrow)>0 else 0)
            ndcg5s.append(ndcg_at_k(rels,5))
        ablation.append({"alpha": alpha, "NDCG@5": round(np.mean(ndcg5s),4)})
    abl_df = pd.DataFrame(ablation)
    best_a = abl_df.loc[abl_df["NDCG@5"].idxmax(), "alpha"]
    print(f"Best alpha={best_a}  NDCG@5={abl_df['NDCG@5'].max():.4f}")
    abl_df.to_csv(EXP_DIR/"alpha_ablation.csv", index=False)
    display(abl_df)

    fig, ax = plt.subplots(figsize=(9,5))
    ax.plot(abl_df["alpha"], abl_df["NDCG@5"], "o-", color="#4C72B0", lw=2, ms=7)
    ax.axvline(best_a, color="#DD8452", ls="--", lw=1.5, label=f"Best alpha={best_a}")
    ax.fill_between(abl_df["alpha"], abl_df["NDCG@5"], alpha=0.12, color="#4C72B0")
    ax.set_xlabel("alpha (FAISS weight; 1-alpha = Graph weight)", fontsize=11)
    ax.set_ylabel("Mean NDCG@5", fontsize=11)
    ax.set_title("GraphRAG-BGE: FAISS vs Graph Weight Ablation (NDCG@5)", fontsize=13)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGS_DIR/"fig_alpha_ablation.png", dpi=150)
    plt.show()
    print("Saved -> figures/fig_alpha_ablation.png")

    # PageRank: relevant vs non-relevant
    g_bge = graphrag_df[graphrag_df["Retriever"]=="GraphRAG-BGE"].copy()
    g_bge["PageRank_Norm"] = g_bge["Chunk_ID"].map(chunk_pr_norm).fillna(0)
    gs_rel = g_bge[g_bge["Relevant"]==1]["PageRank_Norm"]
    gs_nr  = g_bge[g_bge["Relevant"]==0]["PageRank_Norm"]
    print(f"\nPageRank  Relevant: mean={gs_rel.mean():.5f}  std={gs_rel.std():.5f}  n={len(gs_rel)}")
    print(f"PageRank  Non-Rel : mean={gs_nr.mean():.5f}  std={gs_nr.std():.5f}  n={len(gs_nr)}")
    if len(gs_rel)>0 and len(gs_nr)>0:
        n_pairs = min(len(gs_rel), len(gs_nr))
        try:
            stat, pval = wilcoxon(gs_rel.values[:n_pairs], gs_nr.values[:n_pairs], alternative="greater")
            print(f"  Wilcoxon one-sided: stat={stat:.2f}, p={pval:.4f}")
        except Exception as e:
            print(f"  Wilcoxon skipped: {e}")

Saved -> figures/fig_faiss_vs_graph_score.png

Running alpha-ablation on GraphRAG-BGE ...



alpha:   0%|          | 0/10 [00:00<?, ?it/s]


alpha:  10%|█         | 1/10 [00:10<01:32, 10.25s/it]


alpha:  20%|██        | 2/10 [00:20<01:21, 10.24s/it]


alpha:  30%|███       | 3/10 [00:30<01:10, 10.08s/it]

...


alpha: 100%|██████████| 10/10 [01:42<00:00, 10.20s/it]

Best alpha=0.1  NDCG@5=0.9930


,alpha,NDCG@5
0,0.1,0.9930
1,0.2,0.9922
2,0.3,0.9922
3,0.4,0.9913
4,0.5,0.9908
5,0.6,0.9908
6,0.7,0.9815
7,0.8,0.9627
8,0.9,0.9441
9,1.0,0.9324


Saved -> figures/fig_alpha_ablation.png



PageRank  Relevant: mean=0.51727  std=0.27508  n=96
PageRank  Non-Rel : mean=0.59185  std=0.27934  n=4
  Wilcoxon one-sided: stat=2.00, p=0.8750


---
## Section 10 - Statistical Significance Testing
Wilcoxon signed-rank tests (each retriever vs BM25 baseline). Bootstrap 95% CIs (N=2000).

In [10]:
# ============================================================
# SECTION 10 -- STATISTICAL SIGNIFICANCE TESTING
# ============================================================
BASELINE  = "BGE"
SIG_ALPHA = 0.05
N_BOOT    = 2000
np.random.seed(42)

per_q_ndcg = {}
for ret in RETRIEVERS:
    per_q_ndcg[ret] = [per_query_ndcg.get(ret,{}).get(q,0.0) for q in EVALUATION_QUERIES]
baseline_vals = np.array(per_q_ndcg[BASELINE])

# Wilcoxon signed-rank test
sig_rows = []
for ret in RETRIEVERS:
    if ret==BASELINE: continue
    rv  = np.array(per_q_ndcg[ret]); bv = baseline_vals
    pairs = [(a,b) for a,b in zip(rv,bv) if a!=b]
    if len(pairs)<5:
        sig_rows.append({"Retriever":ret,"Mean_NDCG@5_Retriever":round(rv.mean(),4),
                         "Wilcoxon_stat":None,"p-value":None,"Significant?":"N/A (<5 diff pairs)"})
        continue
    rvp, bvp = zip(*pairs)
    try:
        stat, pval = wilcoxon(rvp, bvp, alternative="two-sided")
    except Exception:
        stat, pval = None, None
    sig_rows.append({
        "Retriever": ret,
        "Mean_NDCG@5_Retriever": round(rv.mean(),4),
        "Mean_NDCG@5_BGE":      round(baseline_vals.mean(),4),
        "Delta": round(rv.mean()-baseline_vals.mean(),4),
        "Wilcoxon_stat": round(stat,3) if stat is not None else None,
        "p-value": round(pval,4) if pval is not None else None,
        "Significant?": ("Yes (p<0.05)" if pval is not None and pval<SIG_ALPHA else "No"),
        "Direction": ("Better" if rv.mean()>baseline_vals.mean() else "Worse"),
    })
sig_df = pd.DataFrame(sig_rows).sort_values("Mean_NDCG@5_Retriever", ascending=False)
print("=== Wilcoxon Signed-Rank Tests vs BGE (NDCG@5) ===")
display(sig_df)
sig_df.to_csv(EXP_DIR/"significance_tests.csv", index=False)

# Bootstrap CI
boot_rows = []
for ret in RETRIEVERS:
    vals = np.array(per_q_ndcg[ret])
    boot_means = [np.mean(np.random.choice(vals, size=len(vals), replace=True)) for _ in range(N_BOOT)]
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    boot_rows.append({
        "Retriever": ret,
        "NDCG@5_Mean": round(vals.mean(),4),
        "CI_2.5%": round(lo,4),
        "CI_97.5%": round(hi,4),
        "CI_Width": round(hi-lo,4),
    })
boot_df = pd.DataFrame(boot_rows).sort_values("NDCG@5_Mean", ascending=False)
print("\n=== Bootstrap 95% CI for NDCG@5 (N=2000) ===")
display(boot_df)
boot_df.to_csv(EXP_DIR/"bootstrap_ci.csv", index=False)

# CI Plot
fig, ax = plt.subplots(figsize=(13,6))
labels_ci = boot_df["Retriever"].tolist()
means_ci  = boot_df["NDCG@5_Mean"].tolist()
lo_err    = [m-lo for m,lo in zip(means_ci, boot_df["CI_2.5%"].tolist())]
hi_err    = [hi-m for m,hi in zip(means_ci, boot_df["CI_97.5%"].tolist())]
bar_cols  = ["#DD8452" if r==BASELINE else "#4C72B0" for r in labels_ci]
ax.barh(labels_ci[::-1], means_ci[::-1], xerr=[lo_err[::-1],hi_err[::-1]],
        color=bar_cols[::-1], alpha=0.85, capsize=5, height=0.6)
ax.axvline(baseline_vals.mean(), color="#DD8452", ls="--", lw=1.5,
           label=f"BGE mean={baseline_vals.mean():.3f}")
ax.set_xlabel("Mean NDCG@5", fontsize=11)
ax.set_title(f"NDCG@5 with 95% Bootstrap CI ({N_BOOT} resamples)", fontsize=13)
ax.legend(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_bootstrap_ci.png", dpi=150)
plt.show()
print("Saved -> figures/fig_bootstrap_ci.png")

=== Wilcoxon Signed-Rank Tests vs BGE (NDCG@5) ===


,Retriever,Mean_NDCG@5_Retriever,Mean_NDCG@5_BGE,Delta,Wilcoxon_stat,p-value,Significant?,Direction
6,GraphRAG-BGE,0.9922,0.9324,0.0598,0.0,0.0020,Yes (p<0.05),Better
4,GraphRAG-InLegalBERT,0.9447,0.9324,0.0123,40.0,0.2769,No,Better
3,GraphRAG-MiniLM,0.9359,0.9324,0.0036,66.0,0.9399,No,Better
5,GraphRAG-LegalBERT,0.9259,0.9324,-0.0065,72.5,0.8900,No,Worse
0,MiniLM,0.9246,0.9324,-0.0077,63.0,0.8209,No,Worse
1,InLegalBERT,0.8671,0.9324,-0.0653,40.5,0.0984,No,Worse
2,LegalBERT,0.8239,0.9324,-0.1084,22.0,0.0040,Yes (p<0.05),Worse



=== Bootstrap 95% CI for NDCG@5 (N=2000) ===


,Retriever,NDCG@5_Mean,CI_2.5%,CI_97.5%,CI_Width
7,GraphRAG-BGE,0.9922,0.9804,1.0000,0.0196
5,GraphRAG-InLegalBERT,0.9447,0.9002,0.9838,0.0835
4,GraphRAG-MiniLM,0.9359,0.8928,0.9732,0.0804
3,BGE,0.9324,0.8859,0.9681,0.0822
6,GraphRAG-LegalBERT,0.9259,0.8735,0.9694,0.0960
0,MiniLM,0.9246,0.8770,0.9675,0.0904
1,InLegalBERT,0.8671,0.8161,0.9190,0.1029
2,LegalBERT,0.8239,0.7569,0.8830,0.1261


Saved -> figures/fig_bootstrap_ci.png


---
## Section 11 - Model Efficiency & Embedding Statistics
Embedding time, query latency, dimension comparison, corpus coverage.

In [11]:
# ============================================================
# SECTION 11 -- EFFICIENCY & EMBEDDING STATISTICS
# ============================================================
eff_rows = []
for mkey in MODELS.keys():
    mdir      = MODEL_DIRS[mkey]
    stat_path = mdir/"statistics.json"
    st = {}
    if stat_path.exists():
        with open(stat_path) as f: st = json.load(f)
    emb        = all_embeddings.get(mkey, np.array([]))
    emb_scores = eval_df[eval_df["Retriever"]==mkey]["Cosine_Score"].dropna()
    eff_rows.append({
        "Model": mkey, "HuggingFace ID": MODELS[mkey],
        "Embed Dim": int(emb.shape[1]) if emb.ndim==2 else st.get("embedding_dimension","?"),
        "Total Chunks": int(emb.shape[0]) if emb.ndim==2 else st.get("total_chunks","?"),
        "Embed Time (s)": st.get("embedding_time_seconds","?"),
        "Avg Query (ms)": st.get("average_query_time_ms","?"),
        "Min Cosine":  round(emb_scores.min(),4)  if len(emb_scores)>0 else "?",
        "Max Cosine":  round(emb_scores.max(),4)  if len(emb_scores)>0 else "?",
        "Mean Cosine": round(emb_scores.mean(),4) if len(emb_scores)>0 else "?",
        "Std Cosine":  round(emb_scores.std(),4)  if len(emb_scores)>0 else "?",
    })
eff_df = pd.DataFrame(eff_rows)
print("=== Model Efficiency & Embedding Statistics ===")
display(eff_df)
eff_df.to_csv(EXP_DIR/"efficiency_stats.csv", index=False)

# Corpus coverage
total_chunks = len(legal_chunks)
cov_rows = []
for ret in RETRIEVERS:
    n_unique = eval_df[eval_df["Retriever"]==ret]["Chunk_ID"].nunique()
    cov_rows.append({"Retriever":ret,"Unique Chunks Retrieved":n_unique,
                     "Coverage %": round(100*n_unique/total_chunks,2)})
cov_summary = pd.DataFrame(cov_rows).sort_values("Coverage %", ascending=False)
print("\nCorpus Coverage (unique chunks retrieved / total chunks):")
display(cov_summary)
cov_summary.to_csv(EXP_DIR/"corpus_coverage.csv", index=False)

# Efficiency plots
valid_eff = eff_df.copy()
for c in ["Embed Time (s)","Avg Query (ms)","Embed Dim"]:
    valid_eff[c] = pd.to_numeric(valid_eff[c], errors="coerce")
valid_eff = valid_eff.dropna(subset=["Embed Time (s)"])
model_colors = dict(zip(MODELS.keys(), COLORS))
bar_cols_eff = [model_colors.get(m,"gray") for m in valid_eff["Model"]]

fig, axes = plt.subplots(1,3,figsize=(16,5))
axes[0].bar(valid_eff["Model"], valid_eff["Embed Time (s)"], color=bar_cols_eff, edgecolor="white")
axes[0].set_title("Embedding Time (s)"); axes[0].set_ylabel("Seconds"); axes[0].grid(axis="y",alpha=0.3)
axes[1].bar(valid_eff["Model"], valid_eff["Embed Dim"], color=bar_cols_eff, edgecolor="white")
axes[1].set_title("Embedding Dimension"); axes[1].set_ylabel("Dimensions"); axes[1].grid(axis="y",alpha=0.3)
axes[2].bar(valid_eff["Model"], valid_eff["Avg Query (ms)"], color=bar_cols_eff, edgecolor="white")
axes[2].set_title("Avg Query Time (ms)"); axes[2].set_ylabel("ms"); axes[2].grid(axis="y",alpha=0.3)
plt.suptitle("Model Efficiency Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_efficiency.png", dpi=150)
plt.show()
print("Saved -> figures/fig_efficiency.png")

=== Model Efficiency & Embedding Statistics ===


,Model,HuggingFace ID,Embed Dim,Total Chunks,Embed Time (s),Avg Query (ms),Min Cosine,Max Cosine,Mean Cosine,Std Cosine
0,MiniLM,sentence-transformers/all-MiniLM-L6-v2,384,6597,55.75,0.0,0.3050,0.7382,0.5194,0.1033
1,InLegalBERT,law-ai/InLegalBERT,768,6597,318.20,0.0,0.2244,0.6755,0.5185,0.1154
2,LegalBERT,nlpaueb/legal-bert-base-uncased,768,6597,314.59,0.0,0.2413,0.8171,0.5573,0.1442
3,BGE,BAAI/bge-base-en-v1.5,768,6597,298.17,0.0,0.6733,0.8272,0.7514,0.0353



Corpus Coverage (unique chunks retrieved / total chunks):


,Retriever,Unique Chunks Retrieved,Coverage %
3,BGE,100,1.52
7,GraphRAG-BGE,98,1.49
0,MiniLM,97,1.47
4,GraphRAG-MiniLM,94,1.42
5,GraphRAG-InLegalBERT,92,1.39
1,InLegalBERT,85,1.29
6,GraphRAG-LegalBERT,39,0.59
2,LegalBERT,29,0.44


Saved -> figures/fig_efficiency.png


---
## Section 12 - Comprehensive Comparison Visualisations
Grouped bar chart, metrics heatmap, radar chart, per-query NDCG@5 matrix, MRR vs NDCG scatter.

In [12]:
# ============================================================
# SECTION 12 -- COMPREHENSIVE VISUALISATIONS
# ============================================================
plot_df = metrics_df.set_index("Retriever")
if "Queries_Evaluated" in plot_df.columns:
    plot_df = plot_df.drop(columns=["Queries_Evaluated"])

metric_cols = [c for c in ["P@1","P@3","P@5","R@3","R@5","MRR","MAP","NDCG@3","NDCG@5","NDCG@10"] if c in plot_df.columns]

# Fig 1: Grouped bar chart
fig, ax = plt.subplots(figsize=(20,7))
x     = np.arange(len(plot_df))
n_met = len(metric_cols)
width = 0.08
pal   = plt.cm.tab10(np.linspace(0,1,n_met))
for i, (col, clr) in enumerate(zip(metric_cols, pal)):
    ax.bar(x + i*width, plot_df[col], width, label=col, color=clr, edgecolor="white", alpha=0.85)
ax.set_xticks(x + width*n_met/2)
ax.set_xticklabels(plot_df.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("All IR Metrics -- All Retrievers", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8, ncol=5)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_grouped_bar.png", dpi=150)
plt.show()
print("Saved -> figures/fig_grouped_bar.png")

# Fig 2: Metrics heatmap
fig, ax = plt.subplots(figsize=(16,8))
import seaborn as sns
sns.heatmap(plot_df[metric_cols].T, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=0.4, ax=ax, cbar_kws={"label":"Score"}, annot_kws={"size":8})
ax.set_title("IR Metrics Heatmap -- All Retrievers", fontsize=14, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_metrics_heatmap.png", dpi=150)
plt.show()
print("Saved -> figures/fig_metrics_heatmap.png")

# Fig 3: Radar chart
radar_metrics = [c for c in ["P@3","R@3","MRR","MAP","NDCG@5"] if c in plot_df.columns]
radar_rets    = plot_df.index.tolist()[:8]
radar_df      = plot_df.loc[radar_rets, radar_metrics]
angles        = np.linspace(0, 2*np.pi, len(radar_metrics), endpoint=False).tolist()
angles       += angles[:1]

fig, ax = plt.subplots(figsize=(10,10), subplot_kw=dict(polar=True))
cmap_r  = plt.cm.tab10(np.linspace(0,1,len(radar_rets)))
for ret, col in zip(radar_rets, cmap_r):
    vals = radar_df.loc[ret, radar_metrics].tolist()
    vals += [vals[0]]
    ax.plot(angles, vals, "o-", lw=2, label=ret, color=col)
    ax.fill(angles, vals, alpha=0.07, color=col)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_metrics, fontsize=12)
ax.set_ylim(0,1)
ax.set_title("Retriever Comparison -- Radar Chart", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35,1.15), fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_radar_chart.png", dpi=150)
plt.show()
print("Saved -> figures/fig_radar_chart.png")

# Fig 4: Per-query NDCG@5 heatmap
pq_ndcg_df = pd.DataFrame(per_query_ndcg).T
pq_ndcg_df.columns = [c[:38] for c in pq_ndcg_df.columns]
fig, ax = plt.subplots(figsize=(20,9))
sns.heatmap(pq_ndcg_df.astype(float), annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.3, ax=ax, cbar_kws={"label":"NDCG@5"}, annot_kws={"size":7})
ax.set_title("Per-Query NDCG@5 -- All Retrievers", fontsize=14, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=42, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_per_query_ndcg.png", dpi=150)
plt.show()
print("Saved -> figures/fig_per_query_ndcg.png")

# Fig 5: MRR vs NDCG@5 scatter
fig, ax = plt.subplots(figsize=(10,7))
sc_pal = plt.cm.tab10(np.linspace(0,1,len(plot_df)))
for i, (ret, col) in enumerate(zip(plot_df.index, sc_pal)):
    ax.scatter(plot_df.loc[ret,"MRR"], plot_df.loc[ret,"NDCG@5"], color=col, s=160, zorder=5)
    ax.annotate(ret, (plot_df.loc[ret,"MRR"], plot_df.loc[ret,"NDCG@5"]),
                textcoords="offset points", xytext=(6,4), fontsize=9, color=col)
ax.set_xlabel("MRR", fontsize=12); ax.set_ylabel("NDCG@5", fontsize=12)
ax.set_title("MRR vs NDCG@5 -- Retrieval Performance Space", fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR/"fig_mrr_vs_ndcg.png", dpi=150)
plt.show()
print("Saved -> figures/fig_mrr_vs_ndcg.png")

Saved -> figures/fig_grouped_bar.png


Saved -> figures/fig_metrics_heatmap.png


Saved -> figures/fig_radar_chart.png


Saved -> figures/fig_per_query_ndcg.png


Saved -> figures/fig_mrr_vs_ndcg.png


---
## Section 13 - Research Summary & LaTeX Table Export
Best retriever per metric, auto-generated narrative, publication-ready LaTeX table.

In [13]:
# ============================================================
# SECTION 13 -- RESEARCH SUMMARY & LATEX TABLE EXPORT
# ============================================================
metric_cols_all = ["P@1","P@3","P@5","R@3","R@5","F1@3","MRR","MAP","NDCG@3","NDCG@5","NDCG@10"]
avail_cols      = [c for c in metric_cols_all if c in metrics_df.columns]

print("=== BEST RETRIEVER PER METRIC ===")
best_rows = []
for col in avail_cols:
    best_idx = metrics_df[col].idxmax()
    best_ret = metrics_df.loc[best_idx, "Retriever"]
    best_val = metrics_df.loc[best_idx, col]
    best_rows.append({"Metric":col,"Best Retriever":best_ret,"Score":round(best_val,4)})
    print(f"  {col:12s}: {best_ret:30s} = {best_val:.4f}")

best_df = pd.DataFrame(best_rows)
best_df.to_csv(EXP_DIR/"best_per_metric.csv", index=False)

# LaTeX table
latex_cols = [c for c in ["P@1","P@3","R@3","MRR","MAP","NDCG@3","NDCG@5"] if c in metrics_df.columns]
latex_df   = metrics_df[["Retriever"]+latex_cols].sort_values("NDCG@5", ascending=False).copy()

def bold_best(series):
    mx = series.max()
    return series.apply(lambda v: r"\textbf{" + f"{v:.4f}" + r"}" if v==mx else f"{v:.4f}")

latex_fmt = latex_df.copy()
for c in latex_cols:
    latex_fmt[c] = bold_best(latex_df[c])

latex_table = latex_fmt.to_latex(
    index=False, escape=False,
    column_format="l"+"r"*len(latex_cols),
    caption=(
        f"Retrieval model comparison on the LegalIR benchmark (Indian court judgements, "
        f"{len(legal_chunks):,} chunks, {len(EVALUATION_QUERIES)} queries, top-5 retrieval). "
        r"Best values per column are \textbf{bolded}."
    ),
    label="tab:model_comparison",
)
latex_path = EXP_DIR/"research_metrics_table.tex"
with open(latex_path, "w", encoding="utf-8") as f:
    f.write(r"\begin{table*}[htbp]" + "\n")
    f.write(r"\centering" + "\n")
    f.write(latex_table)
    f.write(r"\end{table*}" + "\n")
print(f"\nLaTeX table saved -> {latex_path}")

# Print preview
with open(latex_path) as f:
    preview = f.read()
print("\n--- LaTeX Preview (first 1000 chars) ---")
print(preview[:1000])

# Narrative summary
top_ret    = metrics_df.sort_values("NDCG@5", ascending=False).iloc[0]
second_ret = metrics_df.sort_values("NDCG@5", ascending=False).iloc[1]
bm25_row   = metrics_df[metrics_df["Retriever"]=="BM25"]

lines_sum = [
    "="*60,
    "RESEARCH SUMMARY",
    "="*60,
    f"Dataset       : {len(legal_chunks):,} legal chunks (Indian court judgements)",
    f"Evaluation    : {len(EVALUATION_QUERIES)} queries | {int(labels_df['Relevant'].sum())} relevant labels",
    f"Retrievers    : {len(RETRIEVERS)} ({', '.join(RETRIEVERS)})",
    f"Primary metric: NDCG@5",
    "",
    f"Top performer : {top_ret['Retriever']} "
    f"(NDCG@5={top_ret['NDCG@5']:.4f}, MRR={top_ret['MRR']:.4f}, MAP={top_ret['MAP']:.4f})",
    f"2nd best      : {second_ret['Retriever']} (NDCG@5={second_ret['NDCG@5']:.4f})",
]
if len(bm25_row)>0:
    br = bm25_row.iloc[0]
    lines_sum.append(f"BM25 baseline : NDCG@5={br['NDCG@5']:.4f}")
lines_sum += [
    "",
    "GraphRAG      : Bipartite citation/act graph, PageRank + Jaccard",
    "              : alpha=0.7 (FAISS), beta=0.3 (Graph). Ablation in Section 9.",
    "",
    "Statistics    : Wilcoxon signed-rank vs BM25 baseline",
    f"              : Bootstrap 95% CI ({N_BOOT} resamples)",
    "="*60,
]
summary = "\n".join(lines_sum)
print("\n"+summary)
with open(EXP_DIR/"research_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary)
print(f"\nSummary saved -> {EXP_DIR/'research_summary.txt'}")
print("\n>>> All done. model_comparison.ipynb completed successfully. <<<")

=== BEST RETRIEVER PER METRIC ===
  P@1         : GraphRAG-BGE                   = 1.0000
  P@3         : GraphRAG-BGE                   = 0.9667
  P@5         : GraphRAG-BGE                   = 0.9600
  R@3         : GraphRAG-BGE                   = 0.1806
  R@5         : GraphRAG-BGE                   = 0.3000
  F1@3        : GraphRAG-BGE                   = 0.3033
  MRR         : GraphRAG-BGE                   = 1.0000
  MAP         : GraphRAG-BGE                   = 0.2948
  NDCG@3      : GraphRAG-BGE                   = 0.9735
  NDCG@5      : GraphRAG-BGE                   = 0.9922
  NDCG@10     : GraphRAG-BGE                   = 0.9922

LaTeX table saved -> E:\Legal Finale\data\experiments\research_metrics_table.tex

--- LaTeX Preview (first 1000 chars) ---
\begin{table*}[htbp]
\centering
\begin{table}
\caption{Retrieval model comparison on the LegalIR benchmark (Indian court judgements, 6,597 chunks, 20 queries, top-5 retrieval). Best values per column are \textbf{bolded}.}
\lab